# 04 - Data Cleaning

## Objective

This notebook cleans, standardizes, and validates the 2025 Airline On-Time Performance dataset before statistical analysis, feature engineering, and machine learning.

The data-cleaning process includes:

- Loading the project configuration and managed raw Delta table
- Validating the approved 32-column schema
- Standardizing dates, binary indicators, numerical fields, and categorical text
- Verifying date parsing and converted data types
- Detecting exact and business-key duplicate records
- Measuring and investigating missing values
- Identifying structurally valid null values associated with cancellations and diversions
- Removing incomplete records that cannot support supervised learning
- Validating permitted categorical and binary value domains
- Checking operational relationships between related variables
- Saving the validated dataset as the managed Delta table `flights_clean`

The resulting table serves as the input for statistical analysis and feature engineering.


#### Load project configuration

The raw dataset is read from the managed Unity Catalog table created during data ingestion. The cleaned dataset is written in Delta format to the processed layer of the project Volume.


In [0]:
# Load the project configuration

from __future__ import annotations

from config import project_config as cfg
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Project configuration loaded successfully.")
print(f"Source table: {cfg.RAW_TABLE}")
print(f"Clean table: {cfg.CLEAN_TABLE}")
print(f"Processed layer: {cfg.PROCESSED_PATH}")
print(f"Clean Delta output: {cfg.CLEAN_DELTA_PATH}")
print(f"Prediction target: {cfg.TARGET_COLUMN}")


Project configuration loaded successfully.
Source table: workspace.default.flights_raw
Clean table: workspace.default.flights_clean
Processed layer: /Volumes/workspace/default/flight_delay_capstone/processed
Clean Delta output: /Volumes/workspace/default/flight_delay_capstone/processed/flights_clean
Prediction target: ARR_DEL15


#### Load the raw dataset

The cleaning process begins by loading the `flights_raw` managed Delta table. This table already combines the twelve monthly BTS files for January through December 2025, so the source CSV files do not need to be read again.


In [0]:
# Load the raw flight table

def require_table(table_name: str) -> None:
    """Validate that a required Unity Catalog table exists."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the data-ingestion notebook before continuing."
        )


require_table(cfg.RAW_TABLE)

df_raw: DataFrame = spark.table(cfg.RAW_TABLE)

raw_row_count = df_raw.count()
raw_column_count = len(df_raw.columns)

print("Raw dataset loaded successfully.")
print(f"Total records: {raw_row_count:,}")
print(f"Total columns: {raw_column_count}")


Raw dataset loaded successfully.
Total records: 7,001,619
Total columns: 32


In [0]:
# Preview raw dataset

display(df_raw.limit(10))


QUARTER,MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
1,1,1,1/6/2025 12:00:00 AM,AA,1,JFK,"New York, NY",New York,LAX,"Los Angeles, CA",California,659,4.0,0.0,28.0,9.0,1020,-8.0,0.0,0.0,null,0.0,381.0,369.0,332.0,2475.0,null,null,null,null,null
1,1,1,1/6/2025 12:00:00 AM,AA,10,LAS,"Las Vegas, NV",Nevada,JFK,"New York, NY",New York,2238,-10.0,0.0,14.0,5.0,630,-33.0,0.0,0.0,null,0.0,292.0,269.0,250.0,2248.0,null,null,null,null,null
1,1,1,1/6/2025 12:00:00 AM,AA,1002,MSN,"Madison, WI",Wisconsin,CLT,"Charlotte, NC",North Carolina,644,1.0,0.0,14.0,22.0,1005,0.0,0.0,0.0,null,0.0,141.0,140.0,104.0,708.0,null,null,null,null,null
1,1,1,1/6/2025 12:00:00 AM,AA,1004,PHL,"Philadelphia, PA",Pennsylvania,LAS,"Las Vegas, NV",Nevada,1855,33.0,1.0,97.0,7.0,2134,83.0,1.0,0.0,null,0.0,339.0,389.0,285.0,2176.0,33.0,0.0,50.0,0.0,0.0
1,1,1,1/6/2025 12:00:00 AM,AA,1005,ORD,"Chicago, IL",Illinois,PHX,"Phoenix, AZ",Arizona,830,15.0,1.0,31.0,4.0,1131,8.0,0.0,0.0,null,0.0,241.0,234.0,199.0,1440.0,null,null,null,null,null
1,1,1,1/6/2025 12:00:00 AM,AA,1006,COS,"Colorado Springs, CO",Colorado,DFW,"Dallas/Fort Worth, TX",Texas,1634,136.0,1.0,13.0,5.0,1927,118.0,1.0,0.0,null,0.0,113.0,95.0,77.0,592.0,0.0,0.0,0.0,0.0,118.0
1,1,1,1/6/2025 12:00:00 AM,AA,1006,DFW,"Dallas/Fort Worth, TX",Texas,COS,"Colorado Springs, CO",Colorado,1500,141.0,1.0,17.0,7.0,1554,140.0,1.0,0.0,null,0.0,114.0,113.0,89.0,592.0,140.0,0.0,0.0,0.0,0.0
1,1,1,1/6/2025 12:00:00 AM,AA,1007,CLT,"Charlotte, NC",North Carolina,STL,"St. Louis, MO",Missouri,2042,null,null,null,null,2153,null,null,1.0,B,0.0,131.0,null,null,575.0,null,null,null,null,null
1,1,1,1/6/2025 12:00:00 AM,AA,1008,LGA,"New York, NY",New York,CLT,"Charlotte, NC",North Carolina,1920,43.0,1.0,28.0,40.0,2129,74.0,1.0,0.0,null,0.0,129.0,160.0,92.0,544.0,7.0,0.0,31.0,0.0,36.0
1,1,1,1/6/2025 12:00:00 AM,AA,1009,EGE,"Eagle, CO",Colorado,MIA,"Miami, FL",Florida,1150,155.0,1.0,11.0,51.0,1803,174.0,1.0,0.0,null,0.0,253.0,272.0,210.0,1810.0,41.0,0.0,19.0,0.0,114.0


#### Schema validation

The raw dataset is compared with the approved list of 32 BTS attributes. This validation identifies missing, unexpected, or incorrectly named columns before any cleaning rules are applied.


In [0]:
# Validate approved BTS schema

actual_columns = df_raw.columns

missing_columns = sorted(set(cfg.EXPECTED_RAW_COLUMNS) - set(actual_columns))
unexpected_columns = sorted(set(actual_columns) - set(cfg.EXPECTED_RAW_COLUMNS))

print(f"Expected columns: {len(cfg.EXPECTED_RAW_COLUMNS)}")
print(f"Actual columns: {len(actual_columns)}")

if missing_columns:
    raise ValueError(
        "Schema validation failed. Missing required columns: "
        f"{missing_columns}"
    )

print("All required columns are present.")

if unexpected_columns:
    print(f"Unexpected columns detected: {unexpected_columns}")
else:
    print("No unexpected columns detected.")


Expected columns: 32
Actual columns: 32
All required columns are present.
No unexpected columns detected.


In [0]:
# Review raw schema

schema_rows = [
    (field.name, field.dataType.simpleString(), field.nullable)
    for field in df_raw.schema.fields
]

schema_df = spark.createDataFrame(
    schema_rows,
    ["COLUMN_NAME", "CURRENT_DATA_TYPE", "NULLABLE"],
)

display(schema_df)


COLUMN_NAME,CURRENT_DATA_TYPE,NULLABLE
QUARTER,int,true
MONTH,int,true
DAY_OF_WEEK,int,true
FL_DATE,string,true
OP_UNIQUE_CARRIER,string,true
OP_CARRIER_FL_NUM,int,true
ORIGIN,string,true
ORIGIN_CITY_NAME,string,true
ORIGIN_STATE_NM,string,true
DEST,string,true


#### Data type and text standardization

Several fields were inferred using generic data types during ingestion. The following standardizations are applied:

- `FL_DATE` is converted from a string containing date and time into Spark `DateType`.
- Binary indicator fields are converted to integers.
- Code and location fields are trimmed and standardized.
- Empty categorical values are converted to null.
- Numerical performance variables remain numeric.

The raw dataset is not modified, and no records are removed during this stage.


In [0]:
# Review cleaning column groups from project configuration

print("Binary indicator columns:")
for column_name in cfg.BINARY_INDICATOR_COLUMNS:
    print(f"- {column_name}")

print("Code columns:")
for column_name in cfg.CODE_COLUMNS:
    print(f"- {column_name}")


Binary indicator columns:
- DEP_DEL15
- ARR_DEL15
- CANCELLED
- DIVERTED
Code columns:
- OP_UNIQUE_CARRIER
- ORIGIN
- ORIGIN_STATE_NM
- DEST
- DEST_STATE_NM
- CANCELLATION_CODE


In [0]:
# Standardize data types and text fields

df_clean = df_raw

df_clean = df_clean.withColumn(
    cfg.FLIGHT_DATE_COLUMN,
    F.to_date(
        F.try_to_timestamp(
            F.trim(F.col(cfg.FLIGHT_DATE_COLUMN)),
            F.lit(cfg.FL_DATE_PARSE_FORMAT),
        )
    ),
)

for column_name in cfg.BINARY_INDICATOR_COLUMNS:
    df_clean = df_clean.withColumn(
        column_name,
        F.col(column_name).cast(T.IntegerType()),
    )

for column_name in cfg.INTEGER_COLUMNS:
    df_clean = df_clean.withColumn(
        column_name,
        F.col(column_name).cast(T.IntegerType()),
    )

for column_name in cfg.DOUBLE_COLUMNS:
    df_clean = df_clean.withColumn(
        column_name,
        F.col(column_name).cast(T.DoubleType()),
    )

for column_name in cfg.CODE_COLUMNS:
    cleaned_value = F.upper(F.trim(F.col(column_name)))
    df_clean = df_clean.withColumn(
        column_name,
        F.when(
            cleaned_value.isNull() | (cleaned_value == ""),
            F.lit(None),
        ).otherwise(cleaned_value),
    )

for column_name in cfg.TEXT_COLUMNS:
    cleaned_value = F.trim(F.col(column_name))
    df_clean = df_clean.withColumn(
        column_name,
        F.when(
            cleaned_value.isNull() | (cleaned_value == ""),
            F.lit(None),
        ).otherwise(cleaned_value),
    )

print("Initial type and text standardization completed.")


Initial type and text standardization completed.


#### Date parsing validation

The `FL_DATE` column is converted from a string to a Spark date during the data type standardization stage. This validation checks whether any original non-null date values failed to convert successfully.

Review `DATE_PARSE_FAILURES` in the output below. A value of **0** confirms that all available flight dates were parsed successfully.


In [0]:
# Validate FL_DATE parsing

date_validation = (
    df_raw
    .select(
        F.count("*").alias("TOTAL_ROWS"),
        F.sum(
            F.when(F.col(cfg.FLIGHT_DATE_COLUMN).isNull(), 1).otherwise(0)
        ).alias("ORIGINAL_NULL_DATES"),
        F.sum(
            F.when(
                F.col(cfg.FLIGHT_DATE_COLUMN).isNotNull()
                & F.try_to_timestamp(
                    F.trim(F.col(cfg.FLIGHT_DATE_COLUMN)),
                    F.lit(cfg.FL_DATE_PARSE_FORMAT),
                ).isNull(),
                1,
            ).otherwise(0)
        ).alias("DATE_PARSE_FAILURES"),
    )
)

display(date_validation)


TOTAL_ROWS,ORIGINAL_NULL_DATES,DATE_PARSE_FAILURES
7001619,0,0


#### Schema verification

After applying the initial data type standardization, the schema is verified to confirm that the selected columns have been converted to their intended data types.


In [0]:
# Review standardized column types

converted_schema_rows = [
    (
        field.name,
        field.dataType.simpleString(),
        field.nullable,
    )
    for field in df_clean.select(*cfg.STANDARDIZED_TYPE_COLUMNS).schema.fields
]

converted_schema_df = spark.createDataFrame(
    converted_schema_rows,
    ["COLUMN_NAME", "STANDARDIZED_DATA_TYPE", "NULLABLE"],
)

display(converted_schema_df)


COLUMN_NAME,STANDARDIZED_DATA_TYPE,NULLABLE
FL_DATE,date,true
DEP_DEL15,int,true
ARR_DEL15,int,true
CANCELLED,int,true
DIVERTED,int,true


#### Standardized dataset preview

A sample of the standardized dataset is displayed below to verify that the applied transformations were successful.


In [0]:
# Preview standardized dataset

display(
    df_clean.select(*cfg.CLEANING_PREVIEW_COLUMNS).limit(20)
)


FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,DEP_DEL15,ARR_DEL15,CANCELLED,DIVERTED
2025-01-06,AA,1,JFK,LAX,0,0,0,0
2025-01-06,AA,10,LAS,JFK,0,0,0,0
2025-01-06,AA,1002,MSN,CLT,0,0,0,0
2025-01-06,AA,1004,PHL,LAS,1,1,0,0
2025-01-06,AA,1005,ORD,PHX,1,0,0,0
2025-01-06,AA,1006,COS,DFW,1,1,0,0
2025-01-06,AA,1006,DFW,COS,1,1,0,0
2025-01-06,AA,1007,CLT,STL,null,null,1,0
2025-01-06,AA,1008,LGA,CLT,1,1,0,0
2025-01-06,AA,1009,EGE,MIA,1,1,0,0


#### Duplicate detection

Duplicate records can distort descriptive statistics, bias model training, and produce inaccurate operational insights. Two types of duplicates are evaluated:

- **Exact duplicates** — records in which all variables contain identical values.
- **Business-key duplicates** — records that share the same flight identity based on flight date, airline, flight number, route, and scheduled departure time.

The business key consists of `FL_DATE`, `OP_UNIQUE_CARRIER`, `OP_CARRIER_FL_NUM`, `ORIGIN`, `DEST`, and `CRS_DEP_TIME`.


In [0]:
# Configure duplicate business key

print("Business key configured successfully.")
print("Business key columns:")
for column_name in cfg.BUSINESS_KEY_COLUMNS:
    print(f"- {column_name}")


Business key configured successfully.
Business key columns:
- FL_DATE
- OP_UNIQUE_CARRIER
- OP_CARRIER_FL_NUM
- ORIGIN
- DEST
- CRS_DEP_TIME


#### Exact duplicate analysis

Exact duplicate analysis compares the total number of standardized records with the number of distinct records across all columns. No records are removed during this analysis.


In [0]:
# Analyze exact duplicate records

standardized_row_count = df_clean.count()
distinct_row_count = df_clean.dropDuplicates().count()
exact_duplicate_count = standardized_row_count - distinct_row_count

exact_duplicate_summary = spark.createDataFrame(
    [
        (
            standardized_row_count,
            distinct_row_count,
            exact_duplicate_count,
        )
    ],
    [
        "TOTAL_STANDARDIZED_ROWS",
        "DISTINCT_ROWS",
        "EXACT_DUPLICATE_ROWS",
    ],
)

display(exact_duplicate_summary)


TOTAL_STANDARDIZED_ROWS,DISTINCT_ROWS,EXACT_DUPLICATE_ROWS
7001619,7001619,0


#### Business key duplicate analysis

Business-key duplicate analysis identifies repeated flight identities using the project business key. No records are removed automatically during this stage.


In [0]:
# Analyze business-key duplicate records

business_key_duplicates = (
    df_clean
    .groupBy(*cfg.BUSINESS_KEY_COLUMNS)
    .agg(F.count("*").alias("RECORD_COUNT"))
    .filter(F.col("RECORD_COUNT") > 1)
    .orderBy(F.col("RECORD_COUNT").desc())
)

duplicate_business_key_count = business_key_duplicates.count()

duplicate_summary_row = (
    business_key_duplicates
    .agg(
        F.coalesce(F.sum("RECORD_COUNT"), F.lit(0)).cast("long").alias("TOTAL_DUPLICATE_RECORDS"),
        F.coalesce(F.sum(F.col("RECORD_COUNT") - 1), F.lit(0)).cast("long").alias("ADDITIONAL_RECORDS_BEYOND_FIRST"),
    )
    .first()
)

business_key_summary = spark.createDataFrame(
    [
        (
            int(duplicate_business_key_count),
            int(duplicate_summary_row["TOTAL_DUPLICATE_RECORDS"]),
            int(duplicate_summary_row["ADDITIONAL_RECORDS_BEYOND_FIRST"]),
        )
    ],
    schema="DUPLICATE_BUSINESS_KEYS long, TOTAL_DUPLICATE_RECORDS long, ADDITIONAL_RECORDS_BEYOND_FIRST long",
)

display(business_key_summary)


DUPLICATE_BUSINESS_KEYS,TOTAL_DUPLICATE_RECORDS,ADDITIONAL_RECORDS_BEYOND_FIRST
0,0,0


#### Missing value analysis

Missing values are evaluated across all selected variables to determine whether they represent data-quality issues or expected operational conditions. No records are removed or imputed during this stage.


In [0]:
# Summarize missing values by column

total_rows = df_clean.count()
null_summary_expressions = []

for column_name in df_clean.columns:
    null_summary_expressions.append(
        F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias(column_name)
    )

null_counts_row = df_clean.select(*null_summary_expressions).first()

null_summary_rows = []
for column_name in df_clean.columns:
    null_count = int(null_counts_row[column_name])
    null_percentage = (null_count / total_rows) * 100 if total_rows > 0 else 0.0
    null_summary_rows.append((column_name, null_count, round(null_percentage, 4)))

null_summary_df = spark.createDataFrame(
    null_summary_rows,
    schema="COLUMN_NAME string, NULL_COUNT long, NULL_PERCENTAGE double",
)

display(null_summary_df.orderBy(F.col("NULL_PERCENTAGE").desc(), F.col("COLUMN_NAME")))


COLUMN_NAME,NULL_COUNT,NULL_PERCENTAGE
CANCELLATION_CODE,6898743,98.5307
CARRIER_DELAY,5466981,78.0817
LATE_AIRCRAFT_DELAY,5466981,78.0817
NAS_DELAY,5466981,78.0817
SECURITY_DELAY,5466981,78.0817
WEATHER_DELAY,5466981,78.0817
ACTUAL_ELAPSED_TIME,122135,1.7444
AIR_TIME,122135,1.7444
ARR_DEL15,122135,1.7444
ARR_DELAY,122135,1.7444


In [0]:
# Review columns containing null values

display(null_summary_df.filter(F.col("NULL_COUNT") > 0))


COLUMN_NAME,NULL_COUNT,NULL_PERCENTAGE
DEP_DELAY,98591,1.4081
DEP_DEL15,98591,1.4081
TAXI_OUT,102183,1.4594
TAXI_IN,104608,1.4941
ARR_DELAY,122135,1.7444
ARR_DEL15,122135,1.7444
CANCELLATION_CODE,6898743,98.5307
CRS_ELAPSED_TIME,2,0.0
ACTUAL_ELAPSED_TIME,122135,1.7444
AIR_TIME,122135,1.7444


In [0]:
# Summarize missing-value coverage

missing_value_overview = (
    null_summary_df
    .agg(
        F.sum(F.when(F.col("NULL_COUNT") > 0, 1).otherwise(0)).alias("COLUMNS_WITH_NULLS"),
        F.sum(F.when(F.col("NULL_COUNT") == 0, 1).otherwise(0)).alias("COLUMNS_WITHOUT_NULLS"),
        F.max("NULL_PERCENTAGE").alias("HIGHEST_NULL_PERCENTAGE"),
    )
)

display(missing_value_overview)


COLUMNS_WITH_NULLS,COLUMNS_WITHOUT_NULLS,HIGHEST_NULL_PERCENTAGE
15,17,98.5307


#### Missing value investigation

This section examines the relationship between missing values and the operational status of flights, particularly cancellations and diversions.


In [0]:
# Investigate missing values by flight status

display(
    df_clean.groupBy(cfg.CANCELLED_COLUMN, cfg.DIVERTED_COLUMN).agg(
        F.count("*").alias("TOTAL"),
        F.sum(F.when(F.col(cfg.TARGET_COLUMN).isNull(), 1).otherwise(0)).alias("NULL_TARGET"),
    )
)


CANCELLED,DIVERTED,TOTAL,NULL_TARGET
1,0,102876,102876
0,1,19258,19258
0,0,6879485,1


#### Missing target investigation

Nearly all missing values in `ARR_DEL15` are associated with cancelled or diverted flights. Records that are not cancelled or diverted but still contain a missing target are reviewed separately.


In [0]:
# Inspect remaining missing target records

remaining_missing_target_record = df_clean.filter(
    (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & F.col(cfg.TARGET_COLUMN).isNull()
)

print(
    "Remaining non-cancelled, non-diverted records "
    f"with missing {cfg.TARGET_COLUMN}: {remaining_missing_target_record.count()}"
)

display(remaining_missing_target_record)


Remaining non-cancelled, non-diverted records with missing ARR_DEL15: 1


QUARTER,MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
3,9,6,2025-09-06,YX,5859,JFK,"New York, NY",NEW YORK,MVY,"Martha's Vineyard, MA",MASSACHUSETTS,1209,0.0,0,26.0,null,1333,null,null,0,null,0,84.0,null,null,173.0,null,null,null,null,null


#### Target cleaning decision

Records with a missing target variable (`ARR_DEL15`) for completed flights cannot be used for supervised machine learning and are excluded from the cleaned dataset. Remaining missing values associated with cancellations or diversions are retained.


In [0]:
# Remove incomplete target records

invalid_target_records = df_clean.filter(
    (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & F.col(cfg.TARGET_COLUMN).isNull()
)

print(f"Invalid target records: {invalid_target_records.count()}")

df_clean = df_clean.filter(
    ~(
        (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
        & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
        & F.col(cfg.TARGET_COLUMN).isNull()
    )
)

print(f"Rows after cleaning: {df_clean.count():,}")


Invalid target records: 1
Rows after cleaning: 7,001,618


#### Domain validation

Domain validation verifies that selected categorical and binary variables contain only values permitted by the BTS data dictionary and the project analytical definitions.


In [0]:
# Validate categorical and binary domains

domain_validation_rows = []

for column_name, valid_values in cfg.DOMAIN_RULES.items():
    invalid_count = (
        df_clean
        .filter(F.col(column_name).isNotNull() & ~F.col(column_name).isin(valid_values))
        .count()
    )
    null_count = df_clean.filter(F.col(column_name).isNull()).count()
    observed_values = [
        row[column_name]
        for row in df_clean.select(column_name).distinct().orderBy(column_name).collect()
    ]
    domain_validation_rows.append(
        (column_name, ", ".join(map(str, valid_values)), str(observed_values), invalid_count, null_count)
    )

domain_validation_df = spark.createDataFrame(
    domain_validation_rows,
    schema="COLUMN_NAME string, EXPECTED_VALUES string, OBSERVED_VALUES string, INVALID_RECORDS long, NULL_RECORDS long",
)

display(domain_validation_df)


COLUMN_NAME,EXPECTED_VALUES,OBSERVED_VALUES,INVALID_RECORDS,NULL_RECORDS
QUARTER,"1, 2, 3, 4","[1, 2, 3, 4]",0,0
MONTH,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]",0,0
DAY_OF_WEEK,"1, 2, 3, 4, 5, 6, 7","[1, 2, 3, 4, 5, 6, 7]",0,0
DEP_DEL15,"0, 1","[None, 0, 1]",0,98591
ARR_DEL15,"0, 1","[None, 0, 1]",0,122134
CANCELLED,"0, 1","[0, 1]",0,0
DIVERTED,"0, 1","[0, 1]",0,0


#### Business rule validation

Business-rule validation checks whether related variables are logically consistent with one another.

**Departure delay indicator consistency**

The `DEP_DEL15` indicator should agree with the recorded departure delay (`DEP_DELAY`). Records with null values are evaluated separately and are not counted as violations in this check.


In [0]:
# Validate departure delay consistency

departure_delay_rule_summary = df_clean.select(
    F.count("*").alias("TOTAL_ROWS"),
    F.sum(
        F.when(
            (F.col(cfg.DEPARTURE_DELAY_FLAG_COLUMN) == 1)
            & F.col(cfg.DEPARTURE_DELAY_COLUMN).isNotNull()
            & (F.col(cfg.DEPARTURE_DELAY_COLUMN) < cfg.DELAY_THRESHOLD_MINUTES),
            1,
        ).otherwise(0)
    ).alias("INDICATOR_1_BUT_DELAY_BELOW_15"),
    F.sum(
        F.when(
            (F.col(cfg.DEPARTURE_DELAY_FLAG_COLUMN) == 0)
            & F.col(cfg.DEPARTURE_DELAY_COLUMN).isNotNull()
            & (F.col(cfg.DEPARTURE_DELAY_COLUMN) >= cfg.DELAY_THRESHOLD_MINUTES),
            1,
        ).otherwise(0)
    ).alias("INDICATOR_0_BUT_DELAY_AT_LEAST_15"),
    F.sum(
        F.when(
            F.col(cfg.DEPARTURE_DELAY_FLAG_COLUMN).isNull()
            | F.col(cfg.DEPARTURE_DELAY_COLUMN).isNull(),
            1,
        ).otherwise(0)
    ).alias("ROWS_WITH_NULL_DEPARTURE_DELAY_FIELDS"),
)

display(departure_delay_rule_summary)


TOTAL_ROWS,INDICATOR_1_BUT_DELAY_BELOW_15,INDICATOR_0_BUT_DELAY_AT_LEAST_15,ROWS_WITH_NULL_DEPARTURE_DELAY_FIELDS
7001618,0,0,98591


#### Arrival delay indicator consistency

The `ARR_DEL15` indicator should be logically consistent with the recorded arrival delay (`ARR_DELAY`). Records with null arrival values are evaluated separately because they may correspond to cancelled or diverted flights.


In [0]:
# Validate arrival delay consistency

arrival_delay_rule_summary = df_clean.select(
    F.count("*").alias("TOTAL_ROWS"),
    F.sum(
        F.when(
            (F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1)
            & F.col(cfg.ARRIVAL_DELAY_COLUMN).isNotNull()
            & (F.col(cfg.ARRIVAL_DELAY_COLUMN) < cfg.DELAY_THRESHOLD_MINUTES),
            1,
        ).otherwise(0)
    ).alias("INDICATOR_1_BUT_DELAY_BELOW_15"),
    F.sum(
        F.when(
            (F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 0)
            & F.col(cfg.ARRIVAL_DELAY_COLUMN).isNotNull()
            & (F.col(cfg.ARRIVAL_DELAY_COLUMN) >= cfg.DELAY_THRESHOLD_MINUTES),
            1,
        ).otherwise(0)
    ).alias("INDICATOR_0_BUT_DELAY_AT_LEAST_15"),
    F.sum(
        F.when(
            F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN).isNull()
            | F.col(cfg.ARRIVAL_DELAY_COLUMN).isNull(),
            1,
        ).otherwise(0)
    ).alias("ROWS_WITH_NULL_ARRIVAL_DELAY_FIELDS"),
)

display(arrival_delay_rule_summary)


TOTAL_ROWS,INDICATOR_1_BUT_DELAY_BELOW_15,INDICATOR_0_BUT_DELAY_AT_LEAST_15,ROWS_WITH_NULL_ARRIVAL_DELAY_FIELDS
7001618,0,0,122134


#### Cancellation code consistency

Flights marked as cancelled should have an associated cancellation reason recorded in `CANCELLATION_CODE`. Flights that are not cancelled are expected to have a null cancellation code.


In [0]:
# Validate cancellation code consistency

cancellation_rule_summary = df_clean.select(
    F.count("*").alias("TOTAL_ROWS"),
    F.sum(
        F.when(
            (F.col(cfg.CANCELLED_COLUMN) == 1)
            & F.col(cfg.CANCELLATION_CODE_COLUMN).isNull(),
            1,
        ).otherwise(0)
    ).alias("CANCELLED_WITHOUT_REASON"),
    F.sum(
        F.when(
            (F.col(cfg.CANCELLED_COLUMN) == 0)
            & F.col(cfg.CANCELLATION_CODE_COLUMN).isNotNull(),
            1,
        ).otherwise(0)
    ).alias("NOT_CANCELLED_WITH_REASON"),
)

display(cancellation_rule_summary)


TOTAL_ROWS,CANCELLED_WITHOUT_REASON,NOT_CANCELLED_WITH_REASON
7001618,0,0


#### Diversion consistency

Flights marked as diverted follow a different operational process from normally completed flights. The arrival delay indicator (`ARR_DEL15`) is expected to be unavailable for diverted flights.


In [0]:
# Validate diversion consistency

diversion_rule_summary = df_clean.select(
    F.count("*").alias("TOTAL_ROWS"),
    F.sum(
        F.when(
            (F.col(cfg.DIVERTED_COLUMN) == 1)
            & F.col(cfg.TARGET_COLUMN).isNotNull(),
            1,
        ).otherwise(0)
    ).alias("DIVERTED_WITH_TARGET"),
    F.sum(
        F.when(
            (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
            & (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
            & F.col(cfg.TARGET_COLUMN).isNull(),
            1,
        ).otherwise(0)
    ).alias("NON_DIVERTED_NON_CANCELLED_WITH_NULL_TARGET"),
)

display(diversion_rule_summary)


TOTAL_ROWS,DIVERTED_WITH_TARGET,NON_DIVERTED_NON_CANCELLED_WITH_NULL_TARGET
7001618,0,0


#### Delay-cause variable consistency

The BTS delay-cause variables describe delay minutes attributed to specific causes. This validation checks whether non-delayed flights contain positive delay-cause values. No records are modified during this validation.


#### Cleaning completion

The data-cleaning workflow validates schema, data types, duplicates, missing values, domain constraints, and business rules before producing the cleaned dataset.

Incomplete records with missing target values for completed flights are excluded. Remaining missing values that represent legitimate operational conditions are retained.


#### Save clean dataset to processed layer

The cleaned dataset is stored in Delta format within the processed layer of the project data lake. Saving the cleaned dataset separately preserves the original raw dataset and supports reproducibility throughout the analytical pipeline.


In [0]:
# Save the cleaned dataset to the processed layer

(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .save(cfg.CLEAN_DELTA_PATH)
)

print("Clean dataset saved successfully.")
print(f"Location: {cfg.CLEAN_DELTA_PATH}")
print(f"Total cleaned records: {df_clean.count():,}")


Clean dataset saved successfully.
Location: /Volumes/workspace/default/flight_delay_capstone/processed/flights_clean
Total cleaned records: 7,001,618


In [0]:
# Validate processed Delta copy

df_processed = spark.read.format("delta").load(cfg.CLEAN_DELTA_PATH)

print(f"Processed records: {df_processed.count():,}")

display(df_processed.limit(10))


Processed records: 7,001,618


QUARTER,MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
3,9,1,2025-09-01,AA,1,JFK,"New York, NY",NEW YORK,LAX,"Los Angeles, CA",CALIFORNIA,700,-7.0,0,18.0,7.0,959,-36.0,0,0,null,0,359.0,330.0,305.0,2475.0,null,null,null,null,null
3,9,1,2025-09-01,AA,10,LAX,"Los Angeles, CA",CALIFORNIA,JFK,"New York, NY",NEW YORK,2121,-8.0,0,18.0,22.0,555,-8.0,0,0,null,0,334.0,334.0,294.0,2475.0,null,null,null,null,null
3,9,1,2025-09-01,AA,1002,MSN,"Madison, WI",WISCONSIN,CLT,"Charlotte, NC",NORTH CAROLINA,716,-7.0,0,12.0,7.0,1030,-21.0,0,0,null,0,134.0,120.0,101.0,708.0,null,null,null,null,null
3,9,1,2025-09-01,AA,1003,CLT,"Charlotte, NC",NORTH CAROLINA,MCI,"Kansas City, MO",MISSOURI,1615,34.0,1,20.0,7.0,1736,35.0,1,0,null,0,141.0,142.0,115.0,808.0,0.0,0.0,1.0,0.0,34.0
3,9,1,2025-09-01,AA,1003,MCI,"Kansas City, MO",MISSOURI,CLT,"Charlotte, NC",NORTH CAROLINA,1833,26.0,1,15.0,19.0,2155,26.0,1,0,null,0,142.0,142.0,108.0,808.0,0.0,0.0,0.0,0.0,26.0
3,9,1,2025-09-01,AA,1004,BOS,"Boston, MA",MASSACHUSETTS,DCA,"Washington, DC",VIRGINIA,2030,-10.0,0,16.0,21.0,2218,-10.0,0,0,null,0,108.0,108.0,71.0,399.0,null,null,null,null,null
3,9,1,2025-09-01,AA,1005,DFW,"Dallas/Fort Worth, TX",TEXAS,BDL,"Hartford, CT",CONNECTICUT,705,24.0,1,16.0,4.0,1135,13.0,0,0,null,0,210.0,199.0,179.0,1471.0,null,null,null,null,null
3,9,1,2025-09-01,AA,1007,HSV,"Huntsville, AL",ALABAMA,DFW,"Dallas/Fort Worth, TX",TEXAS,511,-4.0,0,14.0,17.0,714,-2.0,0,0,null,0,123.0,125.0,94.0,603.0,null,null,null,null,null
3,9,1,2025-09-01,AA,1008,ATL,"Atlanta, GA",GEORGIA,DFW,"Dallas/Fort Worth, TX",TEXAS,1632,-5.0,0,50.0,13.0,1755,42.0,1,0,null,0,143.0,190.0,127.0,731.0,0.0,0.0,42.0,0.0,0.0
3,9,1,2025-09-01,AA,1010,DFW,"Dallas/Fort Worth, TX",TEXAS,STL,"St. Louis, MO",MISSOURI,2106,34.0,1,22.0,7.0,2259,23.0,1,0,null,0,113.0,102.0,73.0,550.0,23.0,0.0,0.0,0.0,0.0


#### Save cleaned managed Delta table

The cleaned dataset is saved as a managed Delta table in Unity Catalog so it can be queried through SQL, accessed by downstream notebooks, and governed through catalog permissions.

The original `flights_raw` table remains unchanged.


In [0]:
# Register cleaned dataset as managed Delta table

(
    df_clean.writeTo(cfg.CLEAN_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Cleaned managed Delta table created successfully.")
print(f"Table: {cfg.CLEAN_TABLE}")
print(f"Total records: {spark.table(cfg.CLEAN_TABLE).count():,}")


Cleaned managed Delta table created successfully.
Table: workspace.default.flights_clean
Total records: 7,001,618
